# 🛢️ Oil Spill Detection — Module 1 Training

## ⚠️ BEFORE YOU RUN
> 1. **Enable GPU**: Settings (⚙️) → Accelerator → **GPU T4 x1** → Save
> 2. **Add Kaggle Secret**: Account → Settings → Secrets → `GDRIVE_OAUTH_CREDS`
>    (generated by running `setup_gdrive_oauth.py` locally — see repo)
> 3. Set `GDRIVE_FOLDER_ID` at the top of **Cell 1**

## 📋 How This Works (Save & Run Mode)
- Cell 1 tests Drive connection **before** starting the 12-hour run — fails fast if broken
- Cell 4 training script **auto-uploads** `best_model.pt` to Drive after every val-loss improvement
- Also uploads `last_model.pt` + `train_metrics.csv` every 5 epochs
- **Account 2 resume**: run Cells 1 → 3 → 4 — Drive checkpoint is downloaded automatically

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 — GPU VERIFICATION  (run first, always)
# ═══════════════════════════════════════════════════════════════════════════════
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Fix: Settings (⚙️) → Accelerator → GPU T4 x1 → Save → Factory Reset session."
    )

device_name = torch.cuda.get_device_name(0)
total_vram  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f"✅ GPU : {device_name} ({total_vram:.1f} GB VRAM)")
print(f"   CUDA: {torch.version.cuda}   PyTorch: {torch.__version__}")
print("\n🟢 GPU ready — proceed to Cell 1.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP: Repo · Dependencies · Google Drive Connection Test
#
# ─── CONFIGURE HERE ──────────────────────────────────────────────────────────
# Paste the folder ID from your Google Drive URL:
#   https://drive.google.com/drive/folders/<PASTE_THIS_PART>
GDRIVE_FOLDER_ID = ""   # ← paste your Drive folder ID here

# Kaggle Secret name that holds your OAuth credentials JSON
# (generated once locally by running setup_gdrive_oauth.py)
GDRIVE_SECRET_NAME = "GDRIVE_OAUTH_CREDS"

# Path where credentials JSON will be written inside Kaggle
GDRIVE_CREDS_PATH = "/kaggle/working/gdrive_creds.json"
# ─────────────────────────────────────────────────────────────────────────────
import os, sys, json

# ── 1a. Clone / pull latest repo ─────────────────────────────────────────────
REPO_URL = "https://github.com/Rohith-Sheregar/Oil-Spill-Detection-New.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✅ Repo: {os.getcwd()}")

# ── 1b. Install dependencies ──────────────────────────────────────────────────
print("\n📦 Installing deps...")
!pip install -q segmentation-models-pytorch albumentations scikit-image \
              scipy joblib imagecodecs google-api-python-client google-auth \
              google-auth-oauthlib
print("✅ Dependencies ready.")

# ── 1c. Write OAuth credentials from Kaggle Secret ───────────────────────────
print(f"\n🔑 Loading Google Drive credentials from Kaggle Secret '{GDRIVE_SECRET_NAME}'...")
try:
    from kaggle_secrets import UserSecretsClient
    _secret = UserSecretsClient().get_secret(GDRIVE_SECRET_NAME)
    _info   = json.loads(_secret)  # validate JSON

    # Confirm it has a refresh_token (required for headless use)
    if "refresh_token" not in _info:
        raise ValueError(
            "The secret JSON does not contain a 'refresh_token'.\n"
            "Re-run setup_gdrive_oauth.py locally and paste the new output."
        )

    with open(GDRIVE_CREDS_PATH, "w") as _f:
        _f.write(_secret)
    print(f"✅ Credentials written to {GDRIVE_CREDS_PATH}")
except Exception as _e:
    print(f"⚠️  Could not load secret '{GDRIVE_SECRET_NAME}': {_e}")
    print("   Drive uploads will be DISABLED. Training will still run.")
    GDRIVE_CREDS_PATH = ""

# ── 1d. Google Drive connection test ─────────────────────────────────────────
# Uploads a tiny 2-byte file, verifies it appears in Drive, deletes it.
# This MUST pass before a 12-hour Save & Run is safe to start.
print("\n🧪 Testing Google Drive connection...")

if not GDRIVE_FOLDER_ID:
    print("⚠️  GDRIVE_FOLDER_ID is empty — skipping Drive test.")
    print("   Set GDRIVE_FOLDER_ID at the top of this cell to enable auto-save.")
elif not GDRIVE_CREDS_PATH or not os.path.exists(GDRIVE_CREDS_PATH):
    print("⚠️  No credentials file — skipping Drive test.")
else:
    try:
        import time
        from google.oauth2.credentials import Credentials
        from google.auth.transport.requests import Request
        from googleapiclient.discovery import build as _build
        from googleapiclient.http import MediaFileUpload as _MFU

        # Load OAuth credentials and refresh if needed
        _creds = Credentials.from_authorized_user_info(
            json.loads(open(GDRIVE_CREDS_PATH).read()),
            scopes=["https://www.googleapis.com/auth/drive"],
        )
        if _creds.expired and _creds.refresh_token:
            _creds.refresh(Request())

        _svc = _build("drive", "v3", credentials=_creds, cache_discovery=False)

        # Write a 2-byte test file and upload it
        _test_name = f"_kaggle_drive_test_{int(time.time())}.txt"
        _test_path = f"/kaggle/working/{_test_name}"
        with open(_test_path, "w") as _tf:
            _tf.write("ok")

        _media  = _MFU(_test_path, mimetype="text/plain", resumable=False)
        _result = _svc.files().create(
            body={"name": _test_name, "parents": [GDRIVE_FOLDER_ID]},
            media_body=_media, fields="id,name",
        ).execute()
        _fid = _result["id"]
        print(f"   ✅ Upload OK  → {_test_name}")

        # Verify
        _found = _svc.files().get(fileId=_fid, fields="name").execute()
        print(f"   ✅ Verify OK  → '{_found['name']}' confirmed in Drive")

        # Cleanup
        _svc.files().delete(fileId=_fid).execute()
        os.remove(_test_path)
        print(f"   ✅ Cleanup OK → test file deleted")
        print()
        print("🟢 Google Drive connected! Auto-save is ACTIVE.")
        print(f"   📂 https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}")

    except Exception as _e:
        print(f"❌ Drive test FAILED: {_e}")
        print()
        print("🔧 Common fixes:")
        print("   • Wrong folder ID? Check the URL of your Drive folder.")
        print("   • Refresh token expired? Re-run setup_gdrive_oauth.py locally.")
        print("   • Secret not enabled? Kaggle sidebar 🔒 → toggle GDRIVE_OAUTH_CREDS ON.")
        raise  # Stop here — don't waste 12 hrs with broken Drive

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA SYMLINKS + SANITY CHECK
# ═══════════════════════════════════════════════════════════════════════════════
import glob, os, shutil
from pathlib import Path

INPUT_DIR = "/kaggle/input/datasets/rohithsheregar"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/kaggle/input/datasets" if os.path.exists("/kaggle/input/datasets") else "/kaggle/input"

print(f"📂 Input root: {INPUT_DIR}")
available_dirs = os.listdir(INPUT_DIR)
for d in available_dirs:
    print(f"   └─ {d}")

working_data_dir = "/kaggle/working/data"
if os.path.exists(working_data_dir):
    shutil.rmtree(working_data_dir)

mappings = {
    "train/oil":       lambda n: "oil" in n and not any(k in n for k in ["lookalike", "no", "test"]),
    "train/lookalike": lambda n: "lookalike" in n,
    "train/no_oil":    lambda n: "no" in n and "oil" in n,
    "test/oil":        lambda n: "test" in n,
}

total_tiffs = 0
print("\n🔗 Creating symlinks...")
for subpath, cond in mappings.items():
    matched = [d for d in available_dirs if cond(d.lower())]
    if not matched:
        print(f"   ❌ WARNING: no dataset matched for '{subpath}'")
        continue
    src = os.path.join(INPUT_DIR, matched[0])
    dst = os.path.join(working_data_dir, subpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(src, "**", "*.tif*"), recursive=True))
    total_tiffs += n
    print(f"   ✅ {subpath} → {matched[0]}  ({n} TIFFs)")

print(f"\n📊 Total TIFFs: {total_tiffs}")

print("\n🔍 Dataset sanity check...")
from src.training.zenodo_sos_dataset import discover_sos_pairs
df_oil = discover_sos_pairs(Path("/kaggle/working/data/train/oil"), include_classes=["oil"])
print(f"✅ {len(df_oil)} oil scene pairs found.")
if len(df_oil) == 0:
    raise RuntimeError("❌ No oil pairs found! Check Kaggle dataset attachments.")
r = df_oil.iloc[0]
print(f"   Scene: {r['scene_id']}  |  image: {r['image_path']}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — RESUME CHECKPOINT DETECTION
#
# Priority 1: Download directly from Google Drive (works cross-account!)
# Priority 2: Load from an attached Kaggle Dataset (same-account)
# Priority 3: Fresh start
#
# For Account 2 resuming from Account 1:
#   → Set GDRIVE_FOLDER_ID in Cell 1 (same folder Account 1 wrote to)
#   → Leave KAGGLE_CKPT_DATASET = "" (Drive is the source)
#   → Run this cell — it downloads last_model.pt directly from Drive
# ─────────────────────────────────────────────────────────────────────────────
import os, glob, io

# For same-account Kaggle Dataset resume (leave "" to use Drive)
KAGGLE_CKPT_DATASET = ""           # e.g. "oil-spill-checkpoints"
PREFER_CHECKPOINT   = "last_model.pt"   # or "best_model.pt"

DOWNLOAD_DIR = "/kaggle/working/resume_ckpt"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

RESUME_CKPT = None

# ── Priority 1: Download from Google Drive ────────────────────────────────────
if GDRIVE_FOLDER_ID and GDRIVE_CREDS_PATH and os.path.exists(GDRIVE_CREDS_PATH):
    print("🔍 Checking Google Drive for checkpoints...")
    try:
        from google.oauth2.credentials import Credentials
        from google.auth.transport.requests import Request
        from googleapiclient.discovery import build as _build2
        from googleapiclient.http import MediaIoBaseDownload as _MIBD

        _creds2 = Credentials.from_authorized_user_info(
            json.loads(open(GDRIVE_CREDS_PATH).read()),
            scopes=["https://www.googleapis.com/auth/drive"],
        )
        if _creds2.expired and _creds2.refresh_token:
            _creds2.refresh(Request())
        _svc2 = _build2("drive", "v3", credentials=_creds2, cache_discovery=False)

        # List .pt files in the Drive folder, most-recently-modified first
        _q = f"'{GDRIVE_FOLDER_ID}' in parents and trashed=false and name contains '.pt'"
        _files = _svc2.files().list(
            q=_q, fields="files(id,name,modifiedTime,size)",
            orderBy="modifiedTime desc"
        ).execute().get("files", [])

        if _files:
            print(f"   Found {len(_files)} checkpoint(s) in Drive:")
            for _f in _files:
                _sz = int(_f.get('size', 0)) / (1024**2)
                print(f"     • {_f['name']}  ({_sz:.0f} MB)  modified: {_f['modifiedTime']}")

            # Pick preferred checkpoint, fallback to most recent
            _target = next((_f for _f in _files if _f["name"] == PREFER_CHECKPOINT), _files[0])
            _dest   = os.path.join(DOWNLOAD_DIR, _target["name"])

            print(f"\n   ⬇️  Downloading '{_target['name']}' from Drive...")
            _req  = _svc2.files().get_media(fileId=_target["id"])
            _buf  = io.FileIO(_dest, "wb")
            _dl   = _MIBD(_buf, _req)
            _done = False
            while not _done:
                _status, _done = _dl.next_chunk()
                if _status:
                    print(f"   {int(_status.progress() * 100):3d}%", end="\r")
            print(f"   ✅ Downloaded to: {_dest}")
            RESUME_CKPT = _dest
        else:
            print("   ℹ️  No .pt files in Drive yet — fresh start or check Kaggle Dataset.")
    except Exception as _e:
        print(f"   ⚠️  Drive download failed: {_e}")

# ── Priority 2: Kaggle Dataset (same-account) ────────────────────────────────
if RESUME_CKPT is None and KAGGLE_CKPT_DATASET:
    print(f"\n🔍 Checking Kaggle Dataset: {KAGGLE_CKPT_DATASET}")
    for _d in [f"/kaggle/input/{KAGGLE_CKPT_DATASET}",
               f"/kaggle/input/{KAGGLE_CKPT_DATASET}/checkpoints"]:
        _c = os.path.join(_d, PREFER_CHECKPOINT)
        if os.path.exists(_c):
            RESUME_CKPT = _c
            break
    if RESUME_CKPT is None:
        _pts = glob.glob(f"/kaggle/input/{KAGGLE_CKPT_DATASET}/**/*.pt", recursive=True)
        if _pts:
            RESUME_CKPT = sorted(_pts)[-1]

# ── Result ───────────────────────────────────────────────────────────────────
import json as _json_mod
if RESUME_CKPT:
    import torch
    _ckpt  = torch.load(RESUME_CKPT, map_location="cpu", weights_only=False)
    _epoch = _ckpt.get("epoch", "?")
    _loss  = _ckpt.get("val_loss", "?")
    _miou  = _ckpt.get("val_miou", "?")
    print(f"\n▶  RESUMING FROM : {RESUME_CKPT}")
    print(f"   Saved epoch    : {_epoch}")
    print(f"   Best val_loss  : {_loss}")
    print(f"   Best mIoU      : {_miou}")
    print(f"   ➡️  Will train from epoch {int(_epoch)+1}")
else:
    print("\n🆕 No checkpoint found — fresh start.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — TRAINING  (auto-uploads to Drive after every best epoch)
# ═══════════════════════════════════════════════════════════════════════════════
import time, torch

# ─── SESSION CONFIG ──────────────────────────────────────────────────────────
# Epochs to train THIS session.
# T4 GPU: ~20 epochs safely fits in 12 hrs.
# P100  : ~30 epochs.
EPOCHS_THIS_SESSION = 20

# Pseudo-label cycles:
# 0 = skip (use on all intermediate sessions)
# 5 = run  (FINAL session only, after all epochs done)
PSEUDO_CYCLES = 0
# ─────────────────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("❌ No GPU! Run Cell 0 first.")

print(f"⚡ GPU      : {torch.cuda.get_device_name(0)}")
print(f"📅 Epochs   : {EPOCHS_THIS_SESSION}")
print(f"▶  Resume   : {RESUME_CKPT or 'None (fresh start)'}")
print(f"☁️  Drive    : {'ACTIVE → ' + GDRIVE_FOLDER_ID if GDRIVE_FOLDER_ID else 'DISABLED'}")
print()

resume_flag      = f"--resume {RESUME_CKPT}"          if RESUME_CKPT         else ""
pseudo_flag      = f"--pseudo-cycles {PSEUDO_CYCLES}" if PSEUDO_CYCLES > 0   else "--no-pseudo"
skip_pseudo_flag = "--skip-pseudo-on-resume"          if RESUME_CKPT         else ""
gdrive_flags     = (
    f"--gdrive-folder-id {GDRIVE_FOLDER_ID} --gdrive-credentials {GDRIVE_CREDS_PATH}"
    if GDRIVE_FOLDER_ID and GDRIVE_CREDS_PATH else ""
)

start = time.time()

!python -m src.training.train_module1 \
    --data-root /kaggle/working/data \
    --results-dir /kaggle/working/results/module1 \
    --input-mode full_5band \
    --epochs {EPOCHS_THIS_SESSION} \
    --lr 1e-3 \
    --batch-size 16 \
    --num-workers 2 \
    {resume_flag} \
    {pseudo_flag} \
    {skip_pseudo_flag} \
    {gdrive_flags}

print(f"\n🏁 Done in {(time.time()-start)/3600:.2f} hrs")
print(f"📁 Local : /kaggle/working/results/module1/checkpoints/")
if GDRIVE_FOLDER_ID:
    print(f"☁️  Drive : https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — SESSION SUMMARY + LOCAL OUTPUT COLLECTION
# Run at end of session to see metrics and collect files for the Output tab.
# (Checkpoints are already in Drive if auto-save was active.)
# ═══════════════════════════════════════════════════════════════════════════════
import os, shutil, glob, csv
from pathlib import Path

CHECKPOINT_SRC = "/kaggle/working/results/module1/checkpoints"
METRICS_SRC    = "/kaggle/working/results/module1/metrics"
OUT            = "/kaggle/working/session_output"
os.makedirs(OUT, exist_ok=True)

print("📦 Collecting files for Output tab...")
for pat in [f"{CHECKPOINT_SRC}/*.pt", f"{METRICS_SRC}/*.csv", f"{METRICS_SRC}/*.json"]:
    for src in glob.glob(pat):
        dst     = shutil.copy(src, OUT)
        size_mb = os.path.getsize(dst) / (1024**2)
        print(f"   ✅ {Path(src).name}  ({size_mb:.1f} MB)")

print(f"\n📂 Files at: {OUT}")

# Training summary
csv_p = f"{METRICS_SRC}/train_metrics.csv"
if os.path.exists(csv_p):
    with open(csv_p) as f:
        rows = list(csv.DictReader(f))
    if rows:
        last = rows[-1]
        best = min(rows, key=lambda r: float(r['val_loss']))
        print("\n📊 Training Summary")
        print(f"   {'Epochs logged':<22}: {len(rows)}")
        print(f"   {'Last epoch':<22}: {last['epoch']}")
        print(f"   {'Last val_loss':<22}: {float(last['val_loss']):.4f}")
        print(f"   {'Last mIoU':<22}: {float(last['val_miou']):.4f}")
        print(f"   {'Best val_loss':<22}: {float(best['val_loss']):.4f}  @ epoch {best['epoch']}")
        print(f"   {'Best mIoU':<22}: {float(best['val_miou']):.4f}  @ epoch {best['epoch']}")

if GDRIVE_FOLDER_ID:
    print(f"\n☁️  Checkpoints also auto-saved to Google Drive:")
    print(f"   https://drive.google.com/drive/folders/{GDRIVE_FOLDER_ID}")

---
## 🔄 Account 2 Resume Guide

### What you need on Account 2
- Same Kaggle Secret: `GDRIVE_OAUTH_CREDS` (add via Account → Settings → Secrets)
- Same `GDRIVE_FOLDER_ID` pasted in Cell 1
- Same Kaggle datasets attached (SOS training data)

### Cells to run on Account 2 (in order)

| Cell | Run | Notes |
|------|-----|-------|
| Cell 0 | ✅ | GPU check |
| Cell 1 | ✅ | Same `GDRIVE_FOLDER_ID` → Drive test should pass |
| Cell 2 | ✅ | Data symlinks — attach same datasets |
| Cell 3 | ✅ | Auto-downloads `last_model.pt` from Drive — **no manual transfer needed** |
| Cell 4 | ✅ | Set `EPOCHS_THIS_SESSION`, training resumes from saved epoch |
| Cell 5 | optional | Summary |

### What you do NOT need
- A Kaggle Dataset with checkpoints
- Manual file download / upload between accounts
- Any `KAGGLE_CKPT_DATASET` setting

> Cell 3 downloads the checkpoint directly from Drive using the OAuth token. Cell 4 resumes training from that exact epoch and continues auto-uploading back to the same Drive folder.